# Problem 3B: DRR bias-adjusted corporate-bond factor analysis

# Data loading

This block defines the sample, loads the raw DRR bond panel, JKP characteristics, and DRR public bias-adjusted factor files, and audits the required inputs.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from IPython.display import display

PROJECT_DIR = Path.cwd().parent
DATA_DIR = PROJECT_DIR / '3B Data'
OUTPUT_DIR = PROJECT_DIR / '3B Outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DRR_FILE = DATA_DIR / 'DRR_Monthly.csv'
JKP_FILE = DATA_DIR / 'JKP_Monthly_sic.csv'
WF_FILE = DATA_DIR / 'DRR_within_firm_sort_public' / 'within_firm_exc_all.csv'
SS_FILE = DATA_DIR / 'DRR_single_sort_public' / 'single_sort_exc_all.csv'
START, END = pd.Timestamp('2002-09-30'), pd.Timestamp('2023-12-31')
T_SAMPLE = len(pd.date_range(START, END, freq="ME"))
# Adapted from DFPS and DMR: both report Newey--West inference with three monthly lags.
HAC_LAGS = 3
RNG = np.random.default_rng(20260910)
assert all(p.exists() for p in [DRR_FILE, JKP_FILE, WF_FILE, SS_FILE])
pd.set_option('display.max_columns', 30)
print('Data directory:', DATA_DIR)
print('Pre-specified sample:', START.date(), 'to', END.date())

Data directory: /Users/amartya/Library/CloudStorage/Dropbox/Dr RAP/Education/Warwick/Finance/Practice of Financial Research/Assignment/Problem 3B/3B Data
Pre-specified sample: 2002-09-30 to 2023-12-31


## Input-file audit and bias-adjusted DRR factors

DRR's README states that the public files contain value-weighted long-short factors and that TRACE-based price signals use a one-business-day gap. `val_ipr_wf` is therefore used for DFPS within-firm value. Bond age is not a price-based sorting signal, but the official DRR series is used to keep cleaning and portfolio construction standardized. No sample-dependent sign correction is applied.

<!-- The DRR README states that the supplied factor returns are value-weighted long–short portfolios and that the _wf suffix denotes within-firm construction. It also states that TRACE-based price signals incorporate a one-business-day gap. We use the supplied val_ipr_wf series as the closest available DRR counterpart to DFPS within-firm value and the supplied age series for bond age. These mappings are analytical choices based on the factor definitions, rather than claims made directly in the README. We do not introduce any additional sample-dependent sign correction; consequently, factor orientations remain as supplied or as defined during construction. -->

In [2]:
wf = pd.read_csv(WF_FILE, parse_dates=['date']).set_index('date').sort_index()
ss = pd.read_csv(SS_FILE, parse_dates=['date']).set_index('date').sort_index()
required = {'val_ipr_wf': WF_FILE, 'age': SS_FILE}
for col, path in required.items():
    frame = wf if path == WF_FILE else ss
    assert col in frame.columns, f'{col} missing from {path.name}'

VAL_within = wf['val_ipr_wf'].rename('VAL_within').loc[START:END]
AGE_bond = ss['age'].rename('AGE_bond').loc[START:END]
audit = pd.DataFrame({
    'source': [WF_FILE.name, SS_FILE.name],
    'column': ['val_ipr_wf', 'age'],
    'first_valid': [VAL_within.first_valid_index(), AGE_bond.first_valid_index()],
    'last_valid': [VAL_within.last_valid_index(), AGE_bond.last_valid_index()],
    'months': [VAL_within.count(), AGE_bond.count()]
})
display(audit)

,source,column,first_valid,last_valid,months
0,within_firm_exc_all.csv,val_ipr_wf,2002-09-30,2023-12-31,256
1,single_sort_exc_all.csv,age,2002-09-30,2023-12-31,256


# DRR bias-adjusted data construction

This block completes all factor construction before any assignment question is answered. It applies the documented safeguards against latent implementation bias, look-ahead bias, data-cleaning discretion, sample-dependent sign choices, and methodological uncertainty.


## Bias-control protocol

1. **Latent implementation bias (LIB):** the price-based within-firm value factor is taken from DRR's public bias-corrected factor file. Its TRACE price signals use a one-business-day implementation gap, so the signal price is not reused as the return denominator.
2. **Look-ahead bias (LAB):** no ex-post trimming or full-sample winsorisation is performed. Equity characteristics are lagged one month before they sort bond returns.
3. **Data error and reproducibility risk:** DRR's error-corrected bond data and preformed factors are used, with an explicit file and column audit.
4. **Researcher degrees of freedom:** factor directions, sample dates, value weighting, rating groups, the three-lag HAC convention, and tests are fixed before examining results. IG/NIG and return-definition sensitivity is reported from the supplied DRR files.
5. **Inference and multiplicity:** Newey-West inference, a moving-block bootstrap, bias-adjusted squared Sharpe ratios, GRS spanning tests, and Benjamini-Hochberg adjusted p-values are reported.

The official DRR factor series are the primary specification. Reconstructing an ordinary month-end price sort from the raw panel would intentionally reintroduce the bias that this notebook is designed to avoid.

## Construct the complete factor dataset

CMKT is the beginning-market-value-weighted corporate-bond excess return. The JKP equity characteristics are dated at month-end $t-1$ and merged to bond returns in month $t$. Because these signals do not contain the bond transaction price used in the return denominator, they are not exposed to DRR's shared-price LIB mechanism. No return is trimmed or winsorised after it is realised.

In [3]:
drr_cols = ['date','cusip','permno','spc_rat','mdc_rat','ret_vw','rfret','mcap_s','tmat','cs',
            'lib','igap_bgn','sig_gap']
drr = pd.read_csv(DRR_FILE, usecols=drr_cols, low_memory=False)
drr['month'] = pd.to_datetime(drr['date']).dt.to_period('M').dt.to_timestamp('M')
# Retain one pre-sample month so September 2002 portfolios can use August information.
FORMATION_START = START - pd.offsets.MonthEnd(1)
drr = drr[drr['month'].between(FORMATION_START, END)].copy()
drr = drr.sort_values(['cusip','month'])

# Current-row ratings are observed at the return endpoint. Use only the most recent
# rating observed strictly before the return month when forming rating groups.
drr['rating_num_current'] = drr['spc_rat'].fillna(drr['mdc_rat'])
drr['_rating_source_current'] = drr['month'].where(drr['rating_num_current'].notna())
drr['rating_num'] = drr.groupby('cusip')['rating_num_current'].transform(
    lambda x: x.ffill().shift(1)
)
drr['rating_source_month'] = drr.groupby('cusip')['_rating_source_current'].transform(
    lambda x: x.ffill().shift(1)
)
drr['rating_group'] = pd.cut(drr['rating_num'], [0,7,10,np.inf], labels=['IG+','IG-','SG'])

# Time to maturity is predetermined; recover its value at the previous month-end.
previous_month_end = drr['month'] - pd.offsets.MonthEnd(1)
drr['tmat_formation'] = drr['tmat'] + (drr['month'] - previous_month_end).dt.days / 365.25
drr['bond_excess'] = drr['ret_vw'] - drr['rfret']

def vwmean(values, weights):
    ok = values.notna() & weights.notna() & (weights > 0)
    return np.average(values[ok], weights=weights[ok]) if ok.any() else np.nan

CMKT = drr.groupby('month').apply(
    lambda g: vwmean(g['bond_excess'], g['mcap_s']), include_groups=False
).rename('CMKT')

firm_month = (drr.dropna(subset=['permno','bond_excess','mcap_s'])
    .groupby(['month','permno'])
    .apply(lambda g: pd.Series({
        'firm_ret': vwmean(g['bond_excess'], g['mcap_s']),
        'firm_mv': g.loc[g['mcap_s'].gt(0), 'mcap_s'].sum(),
        'rating_num': vwmean(g['rating_num'], g['mcap_s'])
    }), include_groups=False).reset_index())
firm_month['rating_group'] = pd.cut(firm_month['rating_num'], [0,7,10,np.inf], labels=['IG+','IG-','SG'])

jkp_cols = ['permno','eom','sic','ret_6_1','seas_1_1na','at_me','bev_mev']
jkp = pd.read_csv(JKP_FILE, usecols=jkp_cols, low_memory=False)
jkp['signal_month'] = pd.to_datetime(jkp['eom']).dt.to_period('M').dt.to_timestamp('M')
jkp['month'] = jkp['signal_month'] + pd.offsets.MonthEnd(1)
jkp = jkp.dropna(subset=['permno']).sort_values('signal_month').drop_duplicates(['permno','month'], keep='last')
firm = firm_month.merge(jkp[['permno','month','ret_6_1','seas_1_1na','at_me','bev_mev']],
                        on=['permno','month'], how='left', validate='one_to_one')
# Industry information is also lagged one month and therefore known at portfolio formation.
sic_link = jkp[['permno','month','sic']].drop_duplicates(['permno','month'])
drr = drr.merge(sic_link, on=['permno','month'], how='left', validate='many_to_one')

def rating_tertile_factor(frame, signal):
    rows = []
    use = frame.dropna(subset=['month','rating_group',signal,'firm_ret','firm_mv'])
    for (month, rating), g in use.groupby(['month','rating_group'], observed=True):
        if g[signal].nunique() < 3:
            continue
        lo, hi = g[signal].quantile([1/3,2/3])
        low, high = g[g[signal] <= lo], g[g[signal] >= hi]
        rows.append((month, rating, vwmean(high['firm_ret'],high['firm_mv']) -
                                    vwmean(low['firm_ret'],low['firm_mv'])))
    legs = pd.DataFrame(rows, columns=['month','rating_group','spread'])
    return legs.groupby('month')['spread'].mean()

VAL_firm = rating_tertile_factor(firm, 'bev_mev').rename('VAL_firm')
EQMOM_seasonal = rating_tertile_factor(firm, 'seas_1_1na').rename('EQMOM_seasonal')
# DFPS orientation: long low-leverage firms and short high-leverage firms.
LEV_firm = (-rating_tertile_factor(firm, 'at_me')).rename('LEV_firm')
EQMOM_6_1 = rating_tertile_factor(firm, 'ret_6_1').rename('EQMOM_6_1')

factors = pd.concat([CMKT, VAL_within, VAL_firm, EQMOM_seasonal, LEV_firm, EQMOM_6_1, AGE_bond], axis=1).loc[START:END]
eq9 = factors[['VAL_within','VAL_firm','EQMOM_seasonal','LEV_firm']].dropna()
eq10 = factors[['CMKT','VAL_within','EQMOM_6_1','AGE_bond']].dropna()
display(pd.DataFrame({'first':factors.apply(pd.Series.first_valid_index),
                      'last':factors.apply(pd.Series.last_valid_index),
                      'months':factors.count(), 'missing':factors.isna().sum()}))
print('Eq. (9) common months:', len(eq9), '| Eq. (10) common months:', len(eq10))

,first,last,months,missing
CMKT,2002-09-30,2023-12-31,256,0
VAL_within,2002-09-30,2023-12-31,256,0
VAL_firm,2002-09-30,2023-12-31,256,0
EQMOM_seasonal,2002-09-30,2023-12-31,256,0
LEV_firm,2002-09-30,2023-12-31,256,0
EQMOM_6_1,2002-09-30,2023-12-31,256,0
AGE_bond,2002-09-30,2023-12-31,256,0


Eq. (9) common months: 256 | Eq. (10) common months: 256


## Checkpoint 1: timing, coverage, and implementation safeguards

Before running the asset-pricing tests, this checkpoint verifies that the constructed panel satisfies the pre-specified timing and data-quality rules. It checks the one-month lag applied to JKP signals, the fixed sample period, the use of DRR's public within-firm value factor, factor coverage, and economically plausible monthly return units. Passing these checks does not establish an exact replication of DFPS; it indicates that the analysis contains no obvious timing, coverage, source, or return-unit error.

Credit ratings are taken from the most recent observation strictly before the return month. To address latent implementation bias, the analysis uses DRR's gap-adjusted `val_ipr_wf` factor, while the custom firm-level factors use JKP signals lagged by one month. Following DMR, the credit-spread sorting signal for month \(t\) is the average credit spread from \(t-12\) through \(t-1\). The variables \texttt{lib}, `igap_bgn`, and `sig_gap` are retained as implementation diagnostics rather than used to select observations ex post.


In [4]:
signal_gap = ((jkp['month'].dt.year - jkp['signal_month'].dt.year) * 12
              + jkp['month'].dt.month - jkp['signal_month'].dt.month)
construction_checks = pd.DataFrame({
    'check': [
        'JKP characteristics are lagged exactly one month',
        'Credit ratings used for sorting strictly predate the return month',
        'DRR price-based signals have at least a one-business-day gap',
        'DRR within-firm value is loaded from the public bias-adjusted file',
        'The sample is exactly 2002-09 through 2023-12',
        'Every factor has at least 90% coverage in the fixed sample',
        'Median absolute monthly factor return is below 5%'
    ],
    'passed': [
        len(signal_gap.dropna()) > 0 and signal_gap.dropna().eq(1).all(),
        (drr.loc[drr['rating_num'].notna(), 'rating_source_month'] <
         drr.loc[drr['rating_num'].notna(), 'month']).all(),
        len(drr['sig_gap'].dropna()) > 0 and drr['sig_gap'].dropna().ge(1).all(),
        WF_FILE.name == 'within_firm_exc_all.csv' and 'val_ipr_wf' in wf,
        factors.index.min() == START and factors.index.max() == END,
        factors.notna().mean().min() >= 0.90,
        factors.abs().median().max() < 0.05
    ]
})
display(construction_checks)
assert construction_checks['passed'].all(), 'Stop: at least one construction checkpoint failed.'
rating_age = ((drr['month'].dt.year-drr['rating_source_month'].dt.year)*12
              +drr['month'].dt.month-drr['rating_source_month'].dt.month).dropna()
bias_timing_audit = pd.Series({
    'minimum JKP lag (months)': signal_gap.dropna().min(),
    'minimum rating lag (months)': rating_age.min(),
    'median rating lag (months)': rating_age.median(),
    'minimum DRR price-signal gap (business days)': drr['sig_gap'].dropna().min(),
    'median absolute LIB diagnostic': drr['lib'].abs().median(),
    'median implementation gap (business days)': drr['igap_bgn'].median()
}, name='value')
display(bias_timing_audit.to_frame())
print('Checkpoint 1 passed: timing, source, coverage, and unit checks all passed.')


,check,passed
0,JKP characteristics are lagged exactly one month,True
1,Credit ratings used for sorting strictly preda...,True
2,DRR price-based signals have at least a one-bu...,True
3,DRR within-firm value is loaded from the publi...,True
4,The sample is exactly 2002-09 through 2023-12,True
5,Every factor has at least 90% coverage in the ...,True
6,Median absolute monthly factor return is below 5%,True


,value
minimum JKP lag (months),1.000000
minimum rating lag (months),1.000000
median rating lag (months),1.000000
minimum DRR price-signal gap (business days),1.000000
median absolute LIB diagnostic,0.004076
median implementation gap (business days),1.000000


Checkpoint 1 passed: timing, source, coverage, and unit checks all passed.


## Shared statistical functions

These functions are defined once here so Parts (a)-(d) can be run sequentially without redefining the analysis.

In [5]:
# Adapted from the three-lag Newey--West inference used in DFPS and DMR.
def nw_mean_test(x, lags=HAC_LAGS):
    x = pd.Series(x).dropna()
    fit = sm.OLS(x.to_numpy(), np.ones((len(x),1))).fit(cov_type='HAC', cov_kwds={'maxlags':lags})
    return float(fit.params[0]), float(fit.tvalues[0]), float(fit.pvalues[0])

# Adapted from DFPS, Section 4.4 and Appendix I, following Benjamini and Hochberg (1995).
# Here the adjustment applies to the pre-specified factor family tested in this assignment.
def bh_adjust(p):
    p = np.asarray(p, float); m = len(p); order = np.argsort(p); ranked = p[order]
    adj = np.minimum.accumulate((ranked * m / np.arange(1,m+1))[::-1])[::-1]
    out = np.empty(m); out[order] = np.minimum(adj,1); return out


# Implements the tangency-portfolio squared Sharpe ratio in Lecture 2, slide 34.
def sr2(x):
    x = pd.DataFrame(x).dropna(); mu=x.mean().to_numpy(); V=x.cov().to_numpy()
    return float(mu @ np.linalg.pinv(V) @ mu)

# Adapted from DMR, p. 7 footnote 10, which follows the BKRS adjustment.
def adjusted_sr2(x):
    x=pd.DataFrame(x).dropna(); T,K=x.shape
    return ((T-K-2)/T)*sr2(x)-K/T

# Classical GRS spanning test: adapted from DFPS and Lecture 2, slides 45--46, 58--59, and 71--74.
def grs_spanning(test_assets, benchmark):
    joined=pd.concat([pd.DataFrame(test_assets),pd.DataFrame(benchmark)],axis=1).dropna()
    N=test_assets.shape[1]; K=benchmark.shape[1]; T=len(joined)
    Y=joined.iloc[:,:N].to_numpy(); F=joined.iloc[:,N:].to_numpy()
    X=np.column_stack([np.ones(T),F]); coef=np.linalg.lstsq(X,Y,rcond=None)[0]
    alpha=coef[0]; resid=Y-X@coef; Sigma=resid.T@resid/(T-K-1)
    mu=F.mean(0); VF=np.atleast_2d(np.cov(F,rowvar=False,ddof=1))
    stat=((T-N-K)/N)*(alpha@np.linalg.pinv(Sigma)@alpha)/(1+mu@np.linalg.pinv(VF)@mu)
    return stat, stats.f.sf(stat,N,T-N-K), N, T-N-K, alpha


# Supplementary moving-block bootstrap, adapted conceptually from DMR, p. 12.
# This helper is not an exact implementation of the DFPS or BKRS resampling procedure.
def block_indices(T, block=12, reps=999):
    n=int(np.ceil(T/block)); starts=RNG.integers(0,T-block+1,size=(reps,n))
    return [np.concatenate([np.arange(s,s+block) for s in row])[:T] for row in starts]



## Construction sensitivity across DRR specifications

This section does not search for the best result. It checks whether the two DRR-supplied bond factors change materially across the documented all-bond, IG, and NIG universes and between excess and duration-adjusted returns. Large variation is interpreted as methodological uncertainty, not as permission to select the most favourable specification.

In [6]:
files=[]
for method,folder,prefix in [('single','DRR_single_sort_public','single_sort'),('within','DRR_within_firm_sort_public','within_firm')]:
    for rtype in ['exc','dur']:
        for universe in ['all','ig','nig']:
            path=DATA_DIR/folder/f'{prefix}_{rtype}_{universe}.csv'
            frame=pd.read_csv(path,parse_dates=['date']).set_index('date').loc[START:END]
            suffix = '' if universe=='all' else ('_ig' if universe=='ig' else '_hy')
            target = ('val_ipr_wf' + suffix) if method=='within' else ('age' + suffix)
            if target in frame:
                x=frame[target].dropna(); mean,t,p=nw_mean_test(x)
                files.append([method,rtype,universe,target,len(x),mean,np.sqrt(12)*mean/x.std(ddof=1),t,p])
sensitivity=pd.DataFrame(files,columns=['method','return_type','universe','factor','T','mean','SR_annual','NW_t','p'])
display(sensitivity.set_index(['factor','return_type','universe']).round(4))

,,,method,T,mean,SR_annual,NW_t,p
factor,return_type,universe,,,,,,
age,exc,all,single,256,0.0011,0.5542,2.3124,0.0208
age_ig,exc,ig,single,256,0.0005,0.2774,1.5248,0.1273
age_hy,exc,nig,single,256,0.0013,0.3361,1.8732,0.0610
age,dur,all,single,256,0.0010,0.5185,2.0974,0.0360
age_ig,dur,ig,single,256,0.0008,0.6237,3.1828,0.0015
age_hy,dur,nig,single,256,0.0015,0.3771,1.9863,0.0470
val_ipr_wf,exc,all,within,256,0.0023,1.1266,3.8055,0.0001
val_ipr_wf_ig,exc,ig,within,256,0.0023,1.1442,4.5840,0.0000
val_ipr_wf_hy,exc,nig,within,256,0.0028,0.5178,1.8346,0.0666


# Part (a): factor statistics and Sharpe ratios

This block reports summary statistics, univariate Sharpe ratios and mean tests, multiple-testing-adjusted p-values, multivariate Sharpe ratios, and joint tests that the factor means are zero.

**Lecture methodology.** The tangency-portfolio squared Sharpe ratio, $\mu^{\prime}V^{-1}\mu$, is derived in Lecture 2, slide 34. The joint mean test below uses the Wald-testing logic developed in Lecture 2, slides 46--57, with a HAC covariance estimate to allow heteroskedasticity and serial dependence.


## Summary statistics and individual-factor inference

The individual mean tests are supporting diagnostics, not the answer to Part (b). Their p-values are adjusted jointly because examining several factors creates a multiple-testing problem.

In [7]:
unique_factors = factors[['CMKT','VAL_within','VAL_firm','EQMOM_seasonal','LEV_firm','EQMOM_6_1','AGE_bond']]
rows = []
for name in unique_factors:
    x = unique_factors[name].dropna(); mean,t,p = nw_mean_test(x)
    rows.append([name,len(x),mean,x.std(ddof=1),np.sqrt(12)*mean/x.std(ddof=1),t,p])
factor_stats = pd.DataFrame(rows, columns=['factor','T','mean_monthly','sd_monthly','SR_annual','NW_t','p_raw'])
factor_stats['p_BH'] = bh_adjust(factor_stats['p_raw'])
display(factor_stats.set_index('factor').round(4))

,T,mean_monthly,sd_monthly,SR_annual,NW_t,p_raw,p_BH
factor,,,,,,,
CMKT,256,0.0032,0.0184,0.6061,2.6822,0.0073,0.0202
VAL_within,256,0.0023,0.0070,1.1266,3.8055,0.0001,0.0010
VAL_firm,256,-0.0002,0.0111,-0.0486,-0.2157,0.8292,0.8748
EQMOM_seasonal,256,0.0024,0.0122,0.6680,2.6248,0.0087,0.0202
LEV_firm,256,-0.0001,0.0129,-0.0371,-0.1576,0.8748,0.8748
EQMOM_6_1,256,0.0011,0.0111,0.3315,1.5728,0.1158,0.1621
AGE_bond,256,0.0011,0.0067,0.5542,2.3124,0.0208,0.0363


## Checkpoint 2: qualitative comparison with DFPS factor-return findings

The assignment explicitly warns that the available data cannot exactly reproduce DFPS. Accordingly, the appropriate diagnostic is qualitative: do the economically corresponding factors have the same return direction, and are the strongest DFPS patterns also statistically visible here? DFPS reports within-firm value, enterprise value, equity momentum, and low leverage in its TRACE-sample four-factor model, and adds CMKT, bond age, and six-month equity momentum in the extended-sample specification (DFPS, pp. 5, 25--28). The table below is a pre-specified comparison, not a search over alternative signals or signs.

For `LEV_firm`, a larger `at_me` represents greater leverage. DFPS describes the factor as buying low-leverage firms, so the raw high-minus-low spread is multiplied by $-1$. The reported series is therefore low-minus-high leverage, matching DFPS's economic convention.


In [8]:
fs = factor_stats.set_index('factor')
benchmarks = pd.DataFrame([
    ['VAL_within',       'positive', fs.loc['VAL_within','mean_monthly'] > 0, fs.loc['VAL_within','p_BH'] < .05],
    ['VAL_firm',         'positive', fs.loc['VAL_firm','mean_monthly'] > 0, fs.loc['VAL_firm','p_BH'] < .05],
    ['EQMOM_seasonal',   'positive', fs.loc['EQMOM_seasonal','mean_monthly'] > 0, fs.loc['EQMOM_seasonal','p_BH'] < .05],
    ['LEV_firm',         'positive low-minus-high leverage', fs.loc['LEV_firm','mean_monthly'] > 0, fs.loc['LEV_firm','p_BH'] < .05],
    ['EQMOM_6_1',        'positive', fs.loc['EQMOM_6_1','mean_monthly'] > 0, fs.loc['EQMOM_6_1','p_BH'] < .05],
    ['AGE_bond',         'positive', fs.loc['AGE_bond','mean_monthly'] > 0, fs.loc['AGE_bond','p_BH'] < .05]
], columns=['factor','DFPS qualitative direction','direction_match','significant_after_BH_5pct'])
benchmarks['our_mean_monthly'] = benchmarks['factor'].map(fs['mean_monthly'])
benchmarks['our_BH_p'] = benchmarks['factor'].map(fs['p_BH'])
benchmarks['assessment'] = np.select(
    [benchmarks.direction_match & benchmarks.significant_after_BH_5pct, benchmarks.direction_match],
    ['strong qualitative match','directional match only'], default='does not match')
display(benchmarks.set_index('factor').round(4))
print('Checkpoint 2 conclusion: the DRR-adjusted data provide a partial—not exact—qualitative replication of DFPS factor-return findings.')


,DFPS qualitative direction,direction_match,significant_after_BH_5pct,our_mean_monthly,our_BH_p,assessment
factor,,,,,,
VAL_within,positive,True,True,0.0023,0.0010,strong qualitative match
VAL_firm,positive,False,False,-0.0002,0.8748,does not match
EQMOM_seasonal,positive,True,True,0.0024,0.0202,strong qualitative match
LEV_firm,positive low-minus-high leverage,False,False,-0.0001,0.8748,does not match
EQMOM_6_1,positive,True,False,0.0011,0.1621,directional match only
AGE_bond,positive,True,True,0.0011,0.0363,strong qualitative match


Checkpoint 2 conclusion: the DRR-adjusted data provide a partial—not exact—qualitative replication of DFPS factor-return findings.


## Multivariate Sharpe ratios and joint inference

The factor-level table above reports the requested summary statistics and univariate tests. The following HAC Wald test asks whether all factor means in each four-factor specification are jointly zero; the associated multivariate Sharpe ratio is zero under the same null.

In [9]:
# Annualised multivariate Sharpe ratio; see Lecture 2, slide 34.
def multivariate_sr(x):
    return np.sqrt(12*sr2(x))

# HAC Wald test adapted from Lecture 2, slides 51--57, using the DFPS/DMR three-lag Newey--West convention.
def hac_joint_zero_means(x, lags=HAC_LAGS):
    x=pd.DataFrame(x).dropna(); T,K=x.shape; u=x-x.mean(); S=u.to_numpy().T@u.to_numpy()/T
    for ell in range(1,min(lags,T-1)+1):
        weight=1-ell/(lags+1); gamma=u.iloc[ell:].to_numpy().T@u.iloc[:-ell].to_numpy()/T
        S += weight*(gamma+gamma.T)
    stat=T*x.mean().to_numpy()@np.linalg.pinv(S)@x.mean().to_numpy()
    return stat,K,stats.chi2.sf(stat,K)

part_a=[]
for label,x in [('DFPS Eq. (9)',eq9),('DFPS Eq. (10)',eq10)]:
    stat,df,p=hac_joint_zero_means(x)
    part_a.append([label,len(x),multivariate_sr(x),stat,df,p])
part_a=pd.DataFrame(part_a,columns=['model','T','multivariate_SR_annual','HAC_Wald','df','p_joint_zero_means'])
display(part_a.set_index('model').round(4))

,T,multivariate_SR_annual,HAC_Wald,df,p_joint_zero_means
model,,,,,
DFPS Eq. (9),256,1.6191,34.2008,4,0.0
DFPS Eq. (10),256,1.4922,32.5253,4,0.0


# Part (b): comparison with the corporate-bond market factor

This block formally compares squared Sharpe ratios and tests whether the DFPS nonmarket factors are spanned by CMKT. It then verifies the inference using a moving-block bootstrap.

**Lecture methodology.** Lecture 2, slides 45--46, defines time-series model tests through the joint zero-alpha null; slides 58--59 give the GRS distribution; and slides 71--74 show that the alpha quadratic form equals the improvement in squared tangency-portfolio Sharpe ratios.


## Squared-Sharpe-ratio and CMKT-spanning tests

For each DFPS specification, the null is that the nonmarket factors are spanned by CMKT. Equivalently, their intercepts in time-series regressions on CMKT are jointly zero. Rejection means that the factors expand the **in-sample mean-variance frontier**; it does not by itself prove that their betas price the cross-section or that they are structural economic risks.

In [10]:
specs={
 'DFPS Eq. (9)': factors[['VAL_within','VAL_firm','EQMOM_seasonal','LEV_firm']],
 'DFPS Eq. (10)': factors[['VAL_within','EQMOM_6_1','AGE_bond']]
}
rows=[]
for label,test in specs.items():
    joined=pd.concat([factors[['CMKT']],test],axis=1).dropna()
    benchmark=joined[['CMKT']]; assets=joined[test.columns]
    grs,p,df1,df2,alpha=grs_spanning(assets,benchmark)
    rows.append([label,len(joined),adjusted_sr2(benchmark),adjusted_sr2(joined),
                 adjusted_sr2(joined)-adjusted_sr2(benchmark),grs,df1,df2,p,np.max(np.abs(alpha))])
spanning=pd.DataFrame(rows,columns=['model','T','adj_SR2_CMKT','adj_SR2_full','adj_SR2_gain','GRS_F','df1','df2','p_spanning','max_abs_alpha'])
display(spanning.set_index('model').round(4))

,T,adj_SR2_CMKT,adj_SR2_full,adj_SR2_gain,GRS_F,df1,df2,p_spanning,max_abs_alpha
model,,,,,,,,,
DFPS Eq. (9),256,0.0264,0.2059,0.1796,12.2011,4,251,0.0,0.0031
DFPS Eq. (10),256,0.0264,0.1656,0.1392,12.5788,3,252,0.0,0.0020


## Moving-block-bootstrap confirmation

The GRS test is the formal parametric spanning test. The block bootstrap is a dependence-robust sensitivity check: complete monthly vectors are resampled in 12-month blocks, preserving contemporaneous factor dependence. The null distribution is imposed by centering each nonmarket factor's residual from its CMKT regression.

In [11]:
# Supplementary dependence-robust spanning check, adapted conceptually from DMR, p. 12.
# It complements, but does not replace, the classical GRS test used by DFPS.
def bootstrap_spanning(test, market, reps=999, block=12):
    z=pd.concat([market,test],axis=1).dropna(); m=z.iloc[:,[0]].to_numpy(); Y=z.iloc[:,1:].to_numpy(); T=len(z)
    X=np.column_stack([np.ones(T),m]); coef=np.linalg.lstsq(X,Y,rcond=None)[0]
    # Impose the spanning null by setting every nonmarket intercept to zero.
    # Keep the estimated CMKT slopes and resample centred residuals.
    fitted=m@coef[1:]; resid=Y-X@coef
    null_Y=fitted + (resid-resid.mean(0))
    obs=adjusted_sr2(z)-adjusted_sr2(z.iloc[:,[0]])
    draws=[]
    for idx in block_indices(T,block,reps):
        zb=pd.DataFrame(np.column_stack([m[idx],null_Y[idx]]))
        draws.append(adjusted_sr2(zb)-adjusted_sr2(zb.iloc[:,[0]]))
    draws=np.asarray(draws)
    return obs, (1+np.sum(draws>=obs))/(reps+1), np.quantile(draws,[.025,.5,.975])

boot_rows=[]
for label,test in specs.items():
    obs,p,q=bootstrap_spanning(test,factors[['CMKT']])
    boot_rows.append([label,obs,p,*q])
bootstrap_results=pd.DataFrame(boot_rows,columns=['model','observed_adj_SR2_gain','bootstrap_p','null_q025','null_median','null_q975'])
display(bootstrap_results.set_index('model').round(4))

,observed_adj_SR2_gain,bootstrap_p,null_q025,null_median,null_q975
model,,,,,
DFPS Eq. (9),0.1796,0.003,-0.0139,0.0017,0.0934
DFPS Eq. (10),0.1392,0.001,-0.0110,-0.0002,0.0560


## Conclusion for Part (b)

The cell below states only what the tests establish. A spanning rejection means that the bias-adjusted factors add in-sample mean-variance information beyond CMKT. It does **not** establish cross-sectional pricing, pervasiveness, correct specification, or a structural risk interpretation; those are the separate questions in Parts (c) and (d).

In [12]:
for _,r in spanning.iterrows():
    decision='reject' if r.p_spanning<0.05 else 'do not reject'
    print(f"{r['model']}: {decision} CMKT spanning (GRS p={r.p_spanning:.4f}); "
          f"bias-adjusted monthly squared-Sharpe gain={r.adj_SR2_gain:.4f}.")
print('\nCaveat: the evidence is conditional on DRR-adjusted implementation, the 2002-2023 sample, and the pre-specified construction. Interpret it as an in-sample frontier result, not proof that every factor is a distinct priced systematic risk.')

DFPS Eq. (9): reject CMKT spanning (GRS p=0.0000); bias-adjusted monthly squared-Sharpe gain=0.1796.
DFPS Eq. (10): reject CMKT spanning (GRS p=0.0000); bias-adjusted monthly squared-Sharpe gain=0.1392.

Caveat: the evidence is conditional on DRR-adjusted implementation, the 2002-2023 sample, and the pre-specified construction. Interpret it as an in-sample frontier result, not proof that every factor is a distinct priced systematic risk.


## Checkpoint 3: does the DFPS-style factor set add to CMKT?

This is the paper-level check most closely aligned with DFPS's factor-selection evidence. A positive adjusted squared-Sharpe gain plus rejection of CMKT spanning means the added factors expand the in-sample mean--variance frontier. Lecture 2, slides 71--74, derives the link between joint alpha tests and differences in squared tangency-portfolio Sharpe ratios; slides 45--46 and 58--59 set out the time-series alpha null and GRS test.


In [13]:
frontier_check = spanning[['model','adj_SR2_gain','p_spanning']].copy()
frontier_check['positive_SR2_gain'] = frontier_check['adj_SR2_gain'] > 0
frontier_check['reject_CMKT_spanning_5pct'] = frontier_check['p_spanning'] < .05
display(frontier_check.set_index('model').round(4))
assert frontier_check[['positive_SR2_gain','reject_CMKT_spanning_5pct']].all().all()
print('Checkpoint 3 passed: both reconstructed DFPS sets add mean–variance information beyond CMKT in this sample.')


,adj_SR2_gain,p_spanning,positive_SR2_gain,reject_CMKT_spanning_5pct
model,,,,
DFPS Eq. (9),0.1796,0.0,True,True
DFPS Eq. (10),0.1392,0.0,True,True


Checkpoint 3 passed: both reconstructed DFPS sets add mean–variance information beyond CMKT in this sample.


# Part (c): 32 test portfolios and cross-sectional pricing

This block constructs the DRR test assets, estimates two-pass OLS and GLS cross-sectional regressions, reports risk premia and pricing fit, and measures the time-series pervasiveness of every factor.

**Lecture methodology.** Lecture 3, slides 4--12, develops the two-pass/Fama--MacBeth CSR; slides 33--34 specify OLS and GLS weights and the cross-sectional $\rho^2$ statistic; slides 81--84 explain why robust standard errors matter under misspecification.


## Construct DMR's 32 test portfolios

The test assets comprise five rating, five maturity, ten credit-spread, and twelve Fama--French 12-industry portfolios. Rating and maturity are not transaction-price sorting signals. Industry membership uses issuer SIC information lagged by one month. Following DMR Section 3.3, the credit-spread signal is calculated as the average credit spread from months $t-12$ through $t-1$ and is used to form portfolios whose returns are measured in month $t$. No realised return is subsequently trimmed, winsorised, or used to determine portfolio membership.

In [14]:
def quantile_portfolios(frame, signal, q, prefix):
    rows=[]
    for month,g in frame.groupby('month'):
        g=g.dropna(subset=[signal,'bond_excess','mcap_s']).copy()
        if len(g)<q or g[signal].nunique()<q: continue
        g['bucket']=pd.qcut(g[signal].rank(method='first'),q,labels=False)+1
        for bucket,b in g.groupby('bucket'):
            rows.append((month,f'{prefix}{int(bucket)}',vwmean(b['bond_excess'],b['mcap_s'])))
    return pd.DataFrame(rows,columns=['month','portfolio','return']).pivot(index='month',columns='portfolio',values='return')

rating_ports=quantile_portfolios(drr,'rating_num',5,'RATING_')
maturity_ports=quantile_portfolios(drr,'tmat_formation',5,'MATURITY_')
drr=drr.sort_values(['cusip','month'])
# Adapted from DMR, Section 3.3: average credit spreads from months t-12 through t-1
# are used to sort bonds for the month-t portfolio return.
drr['cs_avg_12_1']=drr.groupby('cusip')['cs'].transform(
    lambda s: s.shift(1).rolling(12, min_periods=12).mean()
)
spread_ports=quantile_portfolios(drr,'cs_avg_12_1',10,'SPREAD_')

def ff12_from_sic(x):
    if pd.isna(x): return np.nan
    s=int(x)
    if (100<=s<=999 or 2000<=s<=2399 or 2700<=s<=2749 or 2770<=s<=2799 or 3100<=s<=3199 or 3940<=s<=3989): return 'NODUR'
    if (2500<=s<=2519 or 2590<=s<=2599 or 3630<=s<=3659 or 3710<=s<=3711 or s in [3714,3716] or 3750<=s<=3751 or s==3792 or 3900<=s<=3939 or 3990<=s<=3999): return 'DURBL'
    if (2520<=s<=2589 or 2600<=s<=2699 or 2750<=s<=2769 or 3000<=s<=3099 or 3200<=s<=3569 or 3580<=s<=3629 or 3700<=s<=3709 or 3712<=s<=3713 or s==3715 or 3717<=s<=3749 or 3752<=s<=3791 or 3793<=s<=3799 or 3830<=s<=3839 or 3860<=s<=3899): return 'MANUF'
    if 1200<=s<=1399 or 2900<=s<=2999: return 'ENERGY'
    if 2800<=s<=2829 or 2840<=s<=2899: return 'CHEMS'
    if 3570<=s<=3579 or 3660<=s<=3692 or 3694<=s<=3699 or 3810<=s<=3829 or 7370<=s<=7379: return 'BUSEQ'
    if 4800<=s<=4899: return 'TELCM'
    if 4900<=s<=4949: return 'UTILS'
    if 5000<=s<=5999 or 7200<=s<=7299 or 7600<=s<=7699: return 'SHOPS'
    if 2830<=s<=2839 or s==3693 or 3840<=s<=3859 or 8000<=s<=8099: return 'HLTH'
    if 6000<=s<=6999: return 'MONEY'
    return 'OTHER'

drr['industry12']=drr['sic'].map(ff12_from_sic)
industry_ports=(drr.dropna(subset=['industry12','bond_excess','mcap_s'])
    .groupby(['month','industry12']).apply(lambda g:vwmean(g['bond_excess'],g['mcap_s']),include_groups=False).unstack())
industry_ports.columns=['IND_'+str(c) for c in industry_ports.columns]
test_assets=pd.concat([rating_ports,maturity_ports,spread_ports,industry_ports],axis=1).loc[START:END]
assert test_assets.shape[1]==32, f'Expected 32 portfolios, found {test_assets.shape[1]}'
coverage=pd.DataFrame({'first':test_assets.apply(pd.Series.first_valid_index),'last':test_assets.apply(pd.Series.last_valid_index),'months':test_assets.count(),'missing':test_assets.isna().sum()})
print('Number of test portfolios:',test_assets.shape[1]); display(coverage)

Number of test portfolios: 32


,first,last,months,missing
RATING_1,2002-09-30,2023-12-31,256,0
RATING_2,2002-09-30,2023-12-31,256,0
RATING_3,2002-09-30,2023-12-31,256,0
RATING_4,2002-09-30,2023-12-31,256,0
RATING_5,2002-09-30,2023-12-31,256,0
MATURITY_1,2002-09-30,2023-12-31,256,0
MATURITY_2,2002-09-30,2023-12-31,256,0
MATURITY_3,2002-09-30,2023-12-31,256,0
MATURITY_4,2002-09-30,2023-12-31,256,0
MATURITY_5,2002-09-30,2023-12-31,256,0


## Two-pass OLS and GLS cross-sectional regressions

The first pass estimates each portfolio's factor betas. The second pass estimates the zero-beta rate and factor risk premia. OLS weights assets equally; GLS uses the inverse test-return covariance matrix. The GLS $R^2$ uses the correctly intercept-adjusted weighted denominator. Risk-premium inference comes from monthly cross-sectional estimates with three-lag Newey--West standard errors, following the convention reported by DFPS and DMR.

In [15]:
# Classical GRS joint-alpha test: adapted from DFPS and Lecture 2, slides 45--46 and 58--59.
def grs_test(R,F):
    z=pd.concat([R,F],axis=1).dropna(); N=R.shape[1]; K=F.shape[1]; T=len(z)
    Y=z[R.columns].to_numpy(); X=np.column_stack([np.ones(T),z[F.columns].to_numpy()])
    coef=np.linalg.lstsq(X,Y,rcond=None)[0]; alpha=coef[0]; resid=Y-X@coef
    Sigma=resid.T@resid/(T-K-1); mf=z[F.columns].mean().to_numpy(); VF=np.atleast_2d(z[F.columns].cov().to_numpy())
    stat=((T-N-K)/N)*(alpha@np.linalg.pinv(Sigma)@alpha)/(1+mf@np.linalg.pinv(VF)@mf)
    return stat,N,T-N-K,stats.f.sf(stat,N,T-N-K)

# Two-pass cross-sectional regression adapted from Lecture 3, slides 4--12 and 33--34.
# Risk-premium inference below uses the three-lag Newey--West convention reported by DFPS and DMR.
def two_pass(R,F,weighting):
    z=pd.concat([R,F],axis=1).dropna(); Rm=z[R.columns]; Fm=z[F.columns]; T,N,K=len(z),R.shape[1],F.shape[1]
    X=np.column_stack([np.ones(T),Fm]); coef=np.linalg.lstsq(X,Rm.to_numpy(),rcond=None)[0]
    betas=coef[1:].T; resid=Rm.to_numpy()-X@coef
    ts_r2=1-(resid**2).sum(0)/((Rm.to_numpy()-Rm.mean().to_numpy())**2).sum(0)
    Z=np.column_stack([np.ones(N),betas]); W=np.eye(N) if weighting=='OLS' else np.linalg.pinv(Rm.cov().to_numpy())
    A=np.linalg.pinv(Z.T@W@Z)@Z.T@W; theta=A@Rm.mean().to_numpy(); errors=Rm.mean().to_numpy()-Z@theta
    one=np.ones(N); G=W-np.outer(W@one,one@W)/(one@W@one)
    csr_r2=1-(errors@W@errors)/(Rm.mean().to_numpy()@G@Rm.mean().to_numpy())
    lambda_t=(A@Rm.to_numpy().T).T
    infer=[nw_mean_test(lambda_t[:,j],HAC_LAGS) for j in range(K+1)]
    grs,df1,df2,pgrs=grs_test(Rm,Fm)
    s=np.linalg.svd(betas-betas.mean(0),compute_uv=False)
    return dict(T=T,N=N,K=K,theta=theta,infer=infer,errors=errors,betas=betas,ts_r2=ts_r2,csr_r2=csr_r2,GRS=grs,GRS_p=pgrs,df1=df1,df2=df2,s=s)

models={'DFPS Eq. (9)':factors[['VAL_within','VAL_firm','EQMOM_seasonal','LEV_firm']],
        'DFPS Eq. (10)':factors[['CMKT','VAL_within','EQMOM_6_1','AGE_bond']]}
results={}; perf=[]; premia=[]
for model,F in models.items():
    for method in ['OLS','GLS']:
        r=two_pass(test_assets,F,method); results[(model,method)]=r
        perf.append([model,method,r['T'],r['N'],r['csr_r2'],np.mean(r['ts_r2']),np.median(r['ts_r2']),r['GRS'],r['GRS_p'],r['s'][-1],r['s'][0]/r['s'][-1]])
        for name,(est,t,p) in zip(['zero_beta']+list(F.columns),r['infer']): premia.append([model,method,name,est,t,p])
model_performance=pd.DataFrame(perf,columns=['model','weighting','T','N','CSR_R2','mean_TS_R2','median_TS_R2','GRS_F','GRS_p','smallest_singular','condition_number'])
risk_premia=pd.DataFrame(premia,columns=['model','weighting','premium','estimate','NW3_t','NW3_p'])
display(model_performance.set_index(['model','weighting']).round(4)); display(risk_premia.set_index(['model','weighting','premium']).round(4))

T   N  CSR_R2  mean_TS_R2  median_TS_R2   GRS_F  \
model         weighting                                                      
DFPS Eq. (9)  OLS        245  32  0.8576      0.2742        0.2174  2.2741   
              GLS        245  32  0.0493      0.2742        0.2174  2.2741   
DFPS Eq. (10) OLS        245  32  0.8921      0.8855        0.9204  2.5887   
              GLS        245  32  0.0956      0.8855        0.9204  2.5887   

                          GRS_p  smallest_singular  condition_number  
model         weighting                                               
DFPS Eq. (9)  OLS        0.0003             1.0160            8.5747  
              GLS        0.0003             1.0160            8.5747  
DFPS Eq. (10) OLS        0.0000             1.1768            3.4785  
              GLS        0.0000             1.1768            3.4785

estimate   NW3_t   NW3_p
model         weighting premium                                 
DFPS Eq. (9)  OLS       zero_beta         0.0010  1.8767  0.0606
                        VAL_within        0.0022  2.0013  0.0454
                        VAL_firm          0.0005  0.7093  0.4782
                        EQMOM_seasonal   -0.0011 -0.7284  0.4664
                        LEV_firm         -0.0011 -1.0936  0.2741
              GLS       zero_beta         0.0006  3.3447  0.0008
                        VAL_within        0.0008  1.1791  0.2384
                        VAL_firm          0.0002  0.2225  0.8239
                        EQMOM_seasonal    0.0004  0.4089  0.6826
                        LEV_firm         -0.0005 -0.5640  0.5728
DFPS Eq. (10) OLS       zero_beta         0.0011  2.4765  0.0133
                        CMKT              0.0018  1.4204  0.1555
                        VAL_within        0.0012  1.5207  0.1283
                        EQMOM_6_1        -0.0008 -0.8722  0.3831
                        AGE_bond          0.0012  1.8443  0.0651
              GLS       zero_beta         0.0006  3.4325  0.0006
                        CMKT              0.0023  1.8645  0.0622
                        VAL_within        0.0007  0.9883  0.3230
                        EQMOM_6_1        -0.0004 -0.5659  0.5715
                        AGE_bond          0.0009  1.5795  0.1142

## Checkpoint 4: cross-sectional pricing versus the DFPS benchmark

DFPS reports that its selected model materially lowers pricing errors, while the assignment asks whether the factors price the 32 DMR portfolios. This checkpoint therefore separates equal-weighted OLS fit from covariance-weighted GLS fit and from the joint zero-alpha specification test. Lecture 3, slides 4--12, introduces Fama--MacBeth/two-pass cross-sectional regressions; slides 33--34 define OLS, GLS, WLS, and the cross-sectional $\rho^2$ measure; slides 81--84 motivate misspecification-robust inference; slides 87--88 discuss formal model comparison.


In [16]:
pricing_check=[]
for _,row in model_performance.iterrows():
    r=results[(row['model'],row['weighting'])]
    nonzero=risk_premia[(risk_premia.model==row['model']) &
                        (risk_premia.weighting==row['weighting']) &
                        (risk_premia.premium!='zero_beta')]
    pricing_check.append([
        row['model'],row['weighting'],row['CSR_R2'],
        np.mean(np.abs(r['errors'])),row['GRS_p'],
        int((nonzero['NW3_p']<.05).sum()),len(nonzero)
    ])
pricing_check=pd.DataFrame(pricing_check,columns=[
    'model','weighting','CSR_R2','mean_abs_pricing_error','GRS_p',
    'significant_factor_premia_5pct','number_of_factor_premia'])
display(pricing_check.set_index(['model','weighting']).round(4))
print('Checkpoint 4 conclusion: OLS fit is high, but GLS fit is much lower and both models fail the joint specification test. Hence the DFPS pricing result is not robustly reproduced on the 32 DMR-style portfolios constructed from DRR data.')


CSR_R2  mean_abs_pricing_error   GRS_p  \
model         weighting                                           
DFPS Eq. (9)  OLS        0.8576                  0.0004  0.0003   
              GLS        0.0493                  0.0019  0.0003   
DFPS Eq. (10) OLS        0.8921                  0.0003  0.0000   
              GLS        0.0956                  0.0004  0.0000   

                         significant_factor_premia_5pct  \
model         weighting                                   
DFPS Eq. (9)  OLS                                     1   
              GLS                                     0   
DFPS Eq. (10) OLS                                     0   
              GLS                                     0   

                         number_of_factor_premia  
model         weighting                           
DFPS Eq. (9)  OLS                              4  
              GLS                              4  
DFPS Eq. (10) OLS                              4  
              GLS                              4

Checkpoint 4 conclusion: OLS fit is high, but GLS fit is much lower and both models fail the joint specification test. Hence the DFPS pricing result is not robustly reproduced on the 32 DMR-style portfolios constructed from DRR data.


## Factor-by-factor time-series pervasiveness


In [17]:
# Factor-pervasiveness diagnostic adapted from DMR: 
# Time-series commonality diagnostic adapted from DMR:
# for each factor, measure how much return variation it explains
# across the 32 test portfolios. Beta dispersion and identification
# are assessed separately using the centred beta-matrix rank test.

def factor_ts_r2(R,f):
    out={}
    for name in R:
        z=pd.concat([R[name],f],axis=1).dropna(); X=sm.add_constant(z.iloc[:,1].to_numpy()); fit=sm.OLS(z.iloc[:,0].to_numpy(),X).fit()
        out[name]=fit.rsquared
    return pd.Series(out)

pervasiveness=pd.DataFrame({name:factor_ts_r2(test_assets,factors[name]) for name in factors.columns})
pervasiveness_summary=pd.DataFrame({'mean_R2':pervasiveness.mean(),'median_R2':pervasiveness.median(),'share_R2_above_10pct':(pervasiveness>.10).mean()})
display(pervasiveness_summary.round(4))



,mean_R2,median_R2,share_R2_above_10pct
CMKT,0.8222,0.8619,1.0000
VAL_within,0.0887,0.0830,0.2188
VAL_firm,0.1626,0.1037,0.5625
EQMOM_seasonal,0.1185,0.0777,0.4375
LEV_firm,0.1542,0.0811,0.4375
EQMOM_6_1,0.1997,0.1553,0.6875
AGE_bond,0.0446,0.0402,0.0312


# Part (d): identification, specification, and systematic-risk assessment

This block tests reduced-rank identification, interprets the GRS specification results already calculated in Part (c), and combines all evidence to determine whether the DFPS factors can be regarded as systematic.

**Lecture methodology.** Reduced-rank identification follows Lecture 4, slides 11, 18--20, and 36. The bond-specific workflow follows Lecture 6, slides 76 and 78. The GRS specification test is from Lecture 2, slides 46 and 58--59.


## Reduced-rank identification test

A factor is pervasive only if it explains common time-series variation across many test portfolios. The first table reports univariate time-series $R^2$ values factor by factor. Identification is assessed from dispersion and rank of the centred beta matrix and with a reduced-rank moving-block bootstrap. The latter is a supplementary diagnostic motivated by Lecture 4, not an exact Cragg--Donald implementation.

In [18]:
# Reduced-rank diagnostic motivated by Lecture 4, slides 11 and 18--20.
# The moving-block bootstrap is a supplementary adaptation, not the exact Cragg--Donald test.
def rank_bootstrap(R,F,reps=499,block=12):
    z=pd.concat([R,F],axis=1).dropna(); Rm=z[R.columns].to_numpy(); Fm=z[F.columns].to_numpy(); T,N,K=len(z),R.shape[1],F.shape[1]
    X=np.column_stack([np.ones(T),Fm]); coef=np.linalg.lstsq(X,Rm,rcond=None)[0]; B=coef[1:].T; Bc=B-B.mean(0)
    U,s,Vt=np.linalg.svd(Bc,full_matrices=False); B0c=(U[:,:K-1]*s[:K-1])@Vt[:K-1]; B0=B0c+B.mean(0)
    alpha0=Rm.mean(0)-Fm.mean(0)@B0.T; resid=Rm-(alpha0+Fm@B0.T); resid-=resid.mean(0)
    null=[]
    for idx in block_indices(T,block,reps):
        Rb=alpha0+Fm@B0.T+resid[idx]; Bb=np.linalg.lstsq(X,Rb,rcond=None)[0][1:].T
        null.append(np.linalg.svd(Bb-Bb.mean(0),compute_uv=False)[-1])
    p=(1+np.sum(np.asarray(null)>=s[-1]))/(reps+1)
    return s[-1],np.quantile(null,.95),p

rank_rows=[]
for model,F in models.items():
    obs,crit,p=rank_bootstrap(test_assets,F); rank_rows.append([model,obs,crit,p,p<.05])
rank_results=pd.DataFrame(rank_rows,columns=['model','smallest_singular','null_95pct','p_reduced_rank','reject_reduced_rank_5pct'])
display(rank_results.set_index('model').round(4))

,smallest_singular,null_95pct,p_reduced_rank,reject_reduced_rank_5pct
model,,,,
DFPS Eq. (9),1.0160,0.6686,0.004,True
DFPS Eq. (10),1.1768,0.4903,0.002,True


## Checkpoint 5: identification before interpretation

A pricing result should not be interpreted if the beta matrix is rank deficient. Lecture 4, slides 11 and 18--20, explains how weak covariance between factors and returns produces reduced rank and spurious inference; slide 36 recommends testing rank before conventional specification and $t$-tests. Lecture 6, slides 76 and 78, applies this logic to the 32 corporate-bond portfolios and summarizes the bond-pricing protocol: construction discipline, economic significance, covariance-versus-beta risk, identification, model misspecification, and robust inference.


In [19]:
identification_check = rank_results[['model','p_reduced_rank','reject_reduced_rank_5pct']].copy()
display(identification_check.set_index('model').round(4))
assert identification_check['reject_reduced_rank_5pct'].all()
print('Checkpoint 5 passed: reduced rank is rejected for both specifications; their weak pricing evidence is therefore not being attributed to an unidentified beta matrix.')


,p_reduced_rank,reject_reduced_rank_5pct
model,,
DFPS Eq. (9),0.004,True
DFPS Eq. (10),0.002,True


Checkpoint 5 passed: reduced rank is rejected for both specifications; their weak pricing evidence is therefore not being attributed to an unidentified beta matrix.


## Overall specification and systematic-risk conclusion

The automated summary below keeps four claims separate: investment-frontier improvement, cross-sectional fit, correct specification, and identification. A high CSR $R^2$ cannot rescue a rejected specification test, and numerical full rank is not enough if the reduced-rank bootstrap cannot reject weak identification. Systematic status additionally requires broad time-series pervasiveness.

In [20]:
print('PART (A)')
for _,r in part_a.iterrows(): print(f"{r['model']}: annualised multivariate SR={r.multivariate_SR_annual:.3f}, joint-zero-means p={r.p_joint_zero_means:.4f}.")
print('\nPART (B)')
for _,r in spanning.iterrows(): print(f"{r['model']}: CMKT spanning p={r.p_spanning:.4g}, adjusted SR2 gain={r.adj_SR2_gain:.3f}.")
print('\nPARTS (C)-(D)')
for _,r in model_performance.iterrows():
    print(f"{r['model']} {r.weighting}: CSR R2={r.CSR_R2:.3f}, mean TS R2={r.mean_TS_R2:.3f}, specification-test p={r.GRS_p:.4g}.")
for _,r in rank_results.iterrows():
    print(f"{r['model']}: reduced-rank p={r.p_reduced_rank:.4f}; reject reduced rank={bool(r.reject_reduced_rank_5pct)}.")
print('\nFinal rule: call a DFPS factor systematic only when it is broadly pervasive, identified, and contributes to a model that is not rejected—not merely because its own mean or the model CSR R2 is large.')

PART (A)
DFPS Eq. (9): annualised multivariate SR=1.619, joint-zero-means p=0.0000.
DFPS Eq. (10): annualised multivariate SR=1.492, joint-zero-means p=0.0000.

PART (B)
DFPS Eq. (9): CMKT spanning p=4.435e-09, adjusted SR2 gain=0.180.
DFPS Eq. (10): CMKT spanning p=1.087e-07, adjusted SR2 gain=0.139.

PARTS (C)-(D)
DFPS Eq. (9) OLS: CSR R2=0.858, mean TS R2=0.274, specification-test p=0.000304.
DFPS Eq. (9) GLS: CSR R2=0.049, mean TS R2=0.274, specification-test p=0.000304.
DFPS Eq. (10) OLS: CSR R2=0.892, mean TS R2=0.885, specification-test p=2.979e-05.
DFPS Eq. (10) GLS: CSR R2=0.096, mean TS R2=0.885, specification-test p=2.979e-05.
DFPS Eq. (9): reduced-rank p=0.0040; reject reduced rank=True.
DFPS Eq. (10): reduced-rank p=0.0020; reject reduced rank=True.

Final rule: call a DFPS factor systematic only when it is broadly pervasive, identified, and contributes to a model that is not rejected—not merely because its own mean or the model CSR R2 is large.


# Exported LaTeX tables and intermediate datasets

Every displayed results table is saved as both a CSV file and a LaTeX fragment. The fragments intentionally omit `\begin{tabular}` and `\end{tabular}` so they can be inserted with `\input{}` inside the document-level table wrapper. Major constructed datasets, beta estimates, and pricing errors are saved as CSV files.

In [21]:
# Export pipeline: LaTeX fragments are designed for \input{} inside an existing tabular environment.
def export_table_fragment(frame, stem, first_column_left=True):
    out = frame.copy()
    out.to_csv(OUTPUT_DIR / f'{stem}.csv', index=False)

    ncols = out.shape[1]
    column_spec = ('l' if first_column_left else 'c') + f'*{{{ncols-1}}}{{c}}'
    full = out.to_latex(
        index=False,
        escape=True,
        na_rep='',
        float_format=lambda value: f'{value:.4f}'
    )
    lines = []
    for line in full.splitlines():
        stripped = line.strip()
        if stripped.startswith(r'\begin{tabular}') or stripped == r'\end{tabular}':
            continue
        if stripped == r'\toprule':
            continue
        lines.append(line)
    fragment = (
        f'% Suggested tabular column specification: {column_spec}\n'
        + '\n'.join(lines)
        + '\n'
    )
    (OUTPUT_DIR / f'{stem}.tex').write_text(fragment, encoding='utf-8')
    return stem, column_spec, len(out), ncols

# Final analytical tables. Each is exported as both .tex and .csv.
tables_to_export = {
    '01_input_file_audit': audit,
    '02_construction_checks': construction_checks,
    '03_drr_construction_sensitivity': sensitivity,
    '04_factor_summary_statistics': factor_stats.rename(columns={
        'factor': 'Factor', 'mean_monthly': 'Mean', 'sd_monthly': 'SD',
        'SR_annual': 'Annualised SR', 'NW_t': 'NW t',
        'p_raw': 'Raw p', 'p_BH': 'BH p'
    }),
    '05_dfps_qualitative_benchmark': benchmarks,
    '06_part_a_joint_factor_tests': part_a,
    '07_part_b_cmkt_spanning': spanning,
    '08_part_b_block_bootstrap': bootstrap_results,
    '09_frontier_checkpoint': frontier_check,
    '10_test_asset_coverage': coverage.rename_axis('portfolio').reset_index(),
    '11_model_performance': model_performance,
    '12_factor_risk_premia': risk_premia,
    '13_pricing_robustness': pricing_check,
    '14_factor_pervasiveness_summary': pervasiveness_summary.rename_axis('factor').reset_index(),
    '15_reduced_rank_results': rank_results,
    '16_identification_checkpoint': identification_check,
}

manifest_rows = []
for stem, table in tables_to_export.items():
    manifest_rows.append(export_table_fragment(table, stem))

# Major intermediate datasets used by the empirical analysis.
factors.rename_axis('month').to_csv(OUTPUT_DIR / 'intermediate_01_factor_returns.csv')
test_assets.rename_axis('month').to_csv(OUTPUT_DIR / 'intermediate_02_dmr_32_test_assets.csv')
pervasiveness.rename_axis('portfolio').to_csv(OUTPUT_DIR / 'intermediate_03_portfolio_factor_r2.csv')

beta_rows = []
error_rows = []
for (model, weighting), result in results.items():
    factor_names = list(models[model].columns)
    for portfolio, beta_vector in zip(test_assets.columns, result['betas']):
        for factor_name, beta_value in zip(factor_names, beta_vector):
            beta_rows.append([model, weighting, portfolio, factor_name, beta_value])
    for portfolio, pricing_error in zip(test_assets.columns, result['errors']):
        error_rows.append([model, weighting, portfolio, pricing_error])

pd.DataFrame(
    beta_rows,
    columns=['model', 'weighting', 'portfolio', 'factor', 'beta']
).to_csv(OUTPUT_DIR / 'intermediate_04_first_pass_betas.csv', index=False)

pd.DataFrame(
    error_rows,
    columns=['model', 'weighting', 'portfolio', 'pricing_error']
).to_csv(OUTPUT_DIR / 'intermediate_05_pricing_errors.csv', index=False)

# A human-readable manifest with the exact \input{} path and tabular specification.
manifest = pd.DataFrame(
    manifest_rows,
    columns=['file_stem', 'suggested_column_spec', 'rows', 'columns']
)
manifest.to_csv(OUTPUT_DIR / '00_output_manifest.csv', index=False)

readme_lines = [
    'Problem 3B output manifest',
    '==========================',
    '',
    'LaTeX table files are fragments: they omit the tabular opening and closing lines.',
    'Use each fragment inside your existing table/threeparttable/tabular wrapper.',
    'Because the required folder name contains a space, the quoted input path is safest.',
    '',
]
for stem, spec, nrows, ncols in manifest_rows:
    readme_lines.extend([
        f'{stem}.tex',
        f'  Suggested tabular: \\begin{{tabular}}{{{spec}}}',
        f'  Input command:     \\input{{"3B Outputs/{stem}"}}',
        f'  Dimensions:        {nrows} rows x {ncols} columns',
        '',
    ])
readme_lines.extend([
    'Intermediate CSV files',
    '----------------------',
    'intermediate_01_factor_returns.csv',
    'intermediate_02_dmr_32_test_assets.csv',
    'intermediate_03_portfolio_factor_r2.csv',
    'intermediate_04_first_pass_betas.csv',
    'intermediate_05_pricing_errors.csv',
    '',
    'PNG files',
    '---------',
    'None: the current notebook does not create figures.',
])
(OUTPUT_DIR / 'README_3B_outputs.txt').write_text('\n'.join(readme_lines) + '\n', encoding='utf-8')

print(f'Exported {len(tables_to_export)} LaTeX table fragments, {len(tables_to_export) + 6} CSV files, and the README manifest to:')
print(OUTPUT_DIR)
display(manifest)


Exported 16 LaTeX table fragments, 22 CSV files, and the README manifest to:
/Users/amartya/Library/CloudStorage/Dropbox/Dr RAP/Education/Warwick/Finance/Practice of Financial Research/Assignment/Problem 3B/3B Outputs


,file_stem,suggested_column_spec,rows,columns
0,01_input_file_audit,l*{4}{c},2,5
1,02_construction_checks,l*{1}{c},7,2
2,03_drr_construction_sensitivity,l*{8}{c},12,9
3,04_factor_summary_statistics,l*{7}{c},7,8
4,05_dfps_qualitative_benchmark,l*{6}{c},6,7
5,06_part_a_joint_factor_tests,l*{5}{c},2,6
6,07_part_b_cmkt_spanning,l*{9}{c},2,10
7,08_part_b_block_bootstrap,l*{5}{c},2,6
8,09_frontier_checkpoint,l*{4}{c},2,5
9,10_test_asset_coverage,l*{4}{c},32,5


# References and methodology map

- Dick-Nielsen, Feldhütter, Pedersen, and Stolborg (DFPS, 2026), *Corporate Bond Factors: Replication Failures and a New Framework*, especially pp. 5, 13, 18--28: qualitative factor findings, factor selection, spanning tests, and pricing-error evidence.
- Dickerson, Robotti, and Rossetti (DRR, 2026), *The Corporate Bond Factor Replication Crisis*: one-business-day price-signal gaps, ex-ante filters, construction discipline, nonstandard errors, and multiple-testing control.
- Dickerson, Müller, and Robotti (DMR, 2023), *Priced Risk in Corporate Bonds*: corporate-bond market benchmark, 32 test portfolios, pervasiveness, identification, and priced-risk distinction.
- **Lecture 2:** slide 34 (tangency squared Sharpe ratio); slides 45--46 (time-series tests and joint alpha null); slides 58--59 (GRS); slides 71--74 (economic meaning as a squared-Sharpe difference).
- **Lecture 3:** slides 4--12 (Fama--MacBeth and two-pass CSR); slides 33--34 (OLS/GLS/WLS and cross-sectional $\\rho^2$); slides 81--84 (robust standard errors); slides 87--88 (model comparison).
- **Lecture 4:** slides 11, 18--20, and 36 (identification failure, reduced-rank tests, spurious factors, and testing order).
- **Lecture 6:** slides 76 and 78 (rank testing on the 32 bond portfolios and the bond asset-pricing protocol).
- *AssignmentPaper2026*, Problem 3, Part B: requested analyses and the instruction to seek qualitative, not exact, replication because the public DFPS files are insufficient for exact reconstruction.
